In [27]:
from datasets import load_dataset, disable_progress_bars
from collections import Counter
disable_progress_bars()

dataset = load_dataset("bitext/Bitext-customer-support-llm-chatbot-training-dataset")

data = dataset["train"]
# cancel_order, change_order, change_shipping_address, track_order, contact_human_agent, get_refund

intents = ["cancel_order", "change_order", "change_shipping_address", "track_order", "contact_human_agent", "get_refund"]
examples_by_intent = {intent: [] for intent in intents}

selected_data = data.filter(lambda row: row["intent"] in intents)

for row in data:
    intent = row["intent"]

    if intent in examples_by_intent and len(examples_by_intent[intent]) < 3:
        examples_by_intent[intent].append(row)

    if all(len(examples_by_intent[intent]) == 3 for intent in intents):
        break

for intent in intents:
    print(f"\n--- {intent} ---")
    for row in examples_by_intent[intent]:
        print(f"Instruction: {row['instruction']}, Intent: {row['intent']}")
from collections import Counter

counts = Counter(data["intent"])



--- cancel_order ---
Instruction: question about cancelling order {{Order Number}}, Intent: cancel_order
Instruction: i have a question about cancelling oorder {{Order Number}}, Intent: cancel_order
Instruction: i need help cancelling puchase {{Order Number}}, Intent: cancel_order

--- change_order ---
Instruction: I try to change several bloody items of order {{Order Number}}, Intent: change_order
Instruction: modifying purchase {{Order Number}}, Intent: change_order
Instruction: want help adding an item to order {{Order Number}}, Intent: change_order

--- change_shipping_address ---
Instruction: give me information about a delivery address modification, Intent: change_shipping_address
Instruction: need to update my address, Intent: change_shipping_address
Instruction: I entered a wrong address, update it, Intent: change_shipping_address

--- track_order ---
Instruction: needx help to check the ETA of purchase {{Order Number}}, Intent: track_order
Instruction: ETA of purchase {{Order

In [28]:
for intent in intents:
    instruction_counts = Counter(
        row["instruction"]
        for row in data
        if row["intent"] == intent
    )

    duplicate_texts = sum(
        1 for count in instruction_counts.values() if count > 1
    )

    extra_duplicate_rows = sum(
        count - 1 for count in instruction_counts.values() if count > 1
    )

    print(
        f"{intent}: {duplicate_texts} duplicate instruction texts, "
        f"{extra_duplicate_rows} extra duplicate rows"
    )

cancel_order: 224 duplicate instruction texts, 505 extra duplicate rows
change_order: 65 duplicate instruction texts, 127 extra duplicate rows
change_shipping_address: 0 duplicate instruction texts, 0 extra duplicate rows
track_order: 111 duplicate instruction texts, 188 extra duplicate rows
contact_human_agent: 0 duplicate instruction texts, 0 extra duplicate rows
get_refund: 21 duplicate instruction texts, 79 extra duplicate rows


In [30]:
from collections import defaultdict

labels_by_instruction = defaultdict(set)

for row in data:
    if row["intent"] in intents:
        instruction = row.get("instruction")

        if instruction is not None and instruction.strip() != "":
            labels_by_instruction[instruction].add(row["intent"])

conflicts = {
    instruction: labels
    for instruction, labels in labels_by_instruction.items()
    if len(labels) > 1
}

print(f"Conflicting instruction texts: {len(conflicts)}")

for instruction, labels in list(conflicts.items())[:10]:
    print(f"{instruction!r} -> {sorted(labels)}")

Conflicting instruction texts: 0


In [31]:
import pandas as pd 
dataset = load_dataset(
    "bitext/Bitext-customer-support-llm-chatbot-training-dataset",
    revision="430d1a89bd93bd1fa23c16f29dd53e73f0087443",
)

dataset = dataset["train"]

intents = [
    "cancel_order",
    "change_order",
    "change_shipping_address",
    "track_order",
    "contact_human_agent",
    "get_refund",
]

selected_data = dataset.filter(lambda row: row["intent"] in intents)
selected_data = selected_data.add_column("row_id", list(range(len(selected_data))))
dataset_df = selected_data.to_pandas()

data_row_id = pd.read_csv(r"C:\Users\alkur\Desktop\Complex Adaptive Systems\Own Projects\support-ticket-assistant\split_ids.csv")

combined_data = pd.merge(dataset_df, data_row_id, on="row_id")

combined_data.shape
combined_data["split"].value_counts()

split
train          4171
validation      596
calibration     596
test            596
Name: count, dtype: int64

In [32]:
Counter(selected_data["intent"])
validation_data = combined_data[combined_data["split"] == "validation"]
validation_data["intent"].value_counts()

correct = (validation_data["intent"] == "contact_human_agent").sum() 
total = validation_data["intent"].value_counts().sum()
accuracy = correct / total
print(correct, total, accuracy)

training_data = combined_data[combined_data["split"] == "train"]
training_data["intent"].value_counts()

100 596 0.16778523489932887


intent
cancel_order               699
contact_human_agent        699
change_order               698
get_refund                 697
track_order                697
change_shipping_address    681
Name: count, dtype: int64

In [33]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report

X_train = training_data["instruction"]
y_train = training_data["intent"]
y_validation = validation_data["intent"]
X_validation = validation_data["instruction"]

pipe = Pipeline([
    ('Tfidf', TfidfVectorizer()),
    ('logistic', LogisticRegression())
])

pipe.fit(X_train, y_train)

y_pred = pipe.predict(X_validation)

accuracy = accuracy_score(y_validation,y_pred)
class_report = classification_report(y_validation, y_pred)
print(accuracy)
print(class_report)

0.9932885906040269
                         precision    recall  f1-score   support

           cancel_order       0.99      1.00      0.99        99
           change_order       0.97      1.00      0.99       100
change_shipping_address       1.00      1.00      1.00        98
    contact_human_agent       1.00      1.00      1.00       100
             get_refund       1.00      1.00      1.00       100
            track_order       1.00      0.96      0.98        99

               accuracy                           0.99       596
              macro avg       0.99      0.99      0.99       596
           weighted avg       0.99      0.99      0.99       596



In [34]:
wrong = y_validation.to_numpy() != y_pred

mistakes = validation_data.loc[wrong, ["instruction", "intent"]].copy()
mistakes["predicted_intent"] = y_pred[wrong]

mistakes

,instruction,intent,predicted_intent
5026,trackorder {{Order Number}},track_order,change_order
5302,trackorder {{Order Number}},track_order,change_order
5704,I need assistance trackng purchase {{Order Num...,track_order,cancel_order
5810,trackorder {{Order Number}},track_order,change_order


In [35]:
import joblib 

joblib.dump(pipe, "artifacts/tfidf_baseline.joblib")

['artifacts/tfidf_baseline.joblib']

In [36]:
test = joblib.load("artifacts/tfidf_baseline.joblib")
test.predict(X_validation) == y_pred

array([ True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,

In [37]:
from transformers import AutoTokenizer, DistilBertModel
import torch


tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
model = DistilBertModel.from_pretrained("distilbert-base-uncased")
text1 = 'I love eating pizza everyday it makes me so happy and beautiful'
text2 = 'I love'
encoded = tokenizer([text1,text2] ,padding="longest")
print(encoded["input_ids"])
print(encoded["attention_mask"])

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 1294.53it/s]
[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[[101, 1045, 2293, 5983, 10733, 10126, 2009, 3084, 2033, 2061, 3407, 1998, 3376, 102], [101, 1045, 2293, 102, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]
[[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]


In [38]:
intent_to_id = {
    "cancel_order": 0,
    "contact_human_agent": 1,
    "change_order": 2,
    "get_refund": 3,
    "track_order": 4,
    "change_shipping_address": 5
}

reverse = {value: key for key, value in intent_to_id.items()}

def id_to_intent(id): 
    return reverse[id]

print(id_to_intent(5))
intent_to_id['change_shipping_address']


change_shipping_address


5

In [40]:
from transformers import AutoConfig, AutoModelForSequenceClassification, DataCollatorWithPadding, Trainer, TrainingArguments
from datasets import Dataset
import evaluate
import numpy as np

def preprocess_function(examples):
    labels = []
    for i in examples["intent"]:
        labels.append(intent_to_id[i])
    tokenizer_dict = tokenizer(examples["instruction"], truncation=True)
    tokenizer_dict.update({'labels': labels})
    return tokenizer_dict

training_data = Dataset.from_pandas(training_data)
validation_data = Dataset.from_pandas(validation_data)

tokenized_training_data = training_data.map(preprocess_function, batched=True)
tokenized_validation_data = validation_data.map(preprocess_function, batched=True)

accuracy = evaluate.load("accuracy")


def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return accuracy.compute(predictions=predictions, references=labels)

model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert/distilbert-base-uncased", num_labels=6, id2label=reverse, label2id=intent_to_id
)

collate_fn = DataCollatorWithPadding(tokenizer, padding=True)

training_args = TrainingArguments(
    output_dir="my_awesome_model",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
)

trainer =Trainer(
    data_collator = collate_fn,
    args=training_args,
    model = model,
    train_dataset=tokenized_training_data,
    eval_dataset=tokenized_validation_data,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,

)

trainer.train()


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 709.04it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
c:\Users\alkur\Desktop\Complex Adaptive Systems\Own Projects\support-ticket-assistant\.venv312\Lib\site-packages\torch\utils\data\dataloader

Epoch,Training Loss,Validation Loss,Accuracy
1,No log,0.028009,1.000000
2,0.224752,0.015976,0.998322


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.01it/s]
c:\Users\alkur\Desktop\Complex Adaptive Systems\Own Projects\support-ticket-assistant\.venv312\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  4.19it/s]


TrainOutput(global_step=522, training_loss=0.21604976058006287, metrics={'train_runtime': 330.6162, 'train_samples_per_second': 25.232, 'train_steps_per_second': 1.579, 'total_flos': 44790888090996.0, 'train_loss': 0.21604976058006287, 'epoch': 2.0})

In [ ]:
trainer.evaluate()

result = trainer.predict(tokenized_validation_data)


c:\Users\alkur\Desktop\Complex Adaptive Systems\Own Projects\support-ticket-assistant\.venv312\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Epoch,Accuracy
0.216373,0.014303,2,0.998322


c:\Users\alkur\Desktop\Complex Adaptive Systems\Own Projects\support-ticket-assistant\.venv312\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


In [ ]:
compare = np.argmax(result.predictions,axis=1) == result.label_ids

mistakes = np.count_nonzero((np.logical_not(compare)))
mistakes


TypeError: 'numpy.int64' object is not iterable

In [ ]:
find_mistakes = np.where(compare == False)
find_mistakes

print(result.predictions[173]) #track_order (wrong) (DistilBERT model)
y_pred[173] # correct

# The TF-IDF got this correct.

[-0.6253143 -1.7915865  2.482944  -2.352927   2.9505854 -2.2346482]


np.str_('change_order')

In [ ]:
# Checking what the DistilBERT got for result where the IF-IDF got wrong 
# Rows where TF-IDF got wrong: 5026, 5302, 5704, 5810

tf_idf_mistakes = [5026,5302,5704,5810]
positions = []
for mistake in tf_idf_mistakes:
    positions.append(validation_data["__index_level_0__"].index(mistake))

positions

[504, 525, 573, 587]

In [ ]:
for position in positions:
    print(result.predictions[position])

# DistilBERT got everything correct that TF-IDF got right

[-1.288226   -1.871093    0.29403487 -2.0419474   4.438778   -1.68209   ]
[-1.288226   -1.871093    0.29403487 -2.0419474   4.438778   -1.68209   ]
[-1.2221634  -1.5948956  -0.78014845 -1.9377934   4.926588   -1.5611122 ]
[-1.288226   -1.871093    0.29403487 -2.0419474   4.438778   -1.68209   ]


In [50]:
# Measuring how much time it takes for DistilBERT 
import time

start = time.perf_counter()
trainer.predict(tokenized_validation_data)
end = time.perf_counter()
end-start

c:\Users\alkur\Desktop\Complex Adaptive Systems\Own Projects\support-ticket-assistant\.venv312\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


8.619194000000789

In [49]:
# Measuring how much time it takes for TF-IDF 
start = time.perf_counter()
pipe.predict(X_validation)
end = time.perf_counter()
end-start


0.02107100000102946

In [51]:
#saving the model
trainer.save_model('artifacts/distilbert_intents')
tokenizer.save_pretrained('artifacts/distilbert_intents')

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.20it/s]


('artifacts/distilbert_intents\\tokenizer_config.json',
 'artifacts/distilbert_intents\\tokenizer.json')

In [2]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
AutoTokenizer.from_pretrained('artifacts/distilbert_intents')
AutoModelForSequenceClassification.from_pretrained('artifacts/distilbert_intents')

Loading weights: 100%|██████████| 104/104 [00:00<00:00, 2446.03it/s]


DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSelfAttention(
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
          (ffn): FFN(
            (dropout): Dropout(